# Создание новых признаков и отбор дескрипторов

После завершения разведочного анализа данных в отдельном ноутбуке на данном этапе была выполнена предобработка, направленная на формирование устойчивого и аналитически обоснованного признакового пространства для последующего построения моделей. Основная цель этого шага состояла в том, чтобы повысить информативность исходных данных и снизить влияние факторов, способных ухудшать качество моделирования.

В качестве исходной базы использовались обучающая и тестовая выборки, содержащие набор молекулярных дескрипторов. Для обучающей выборки дополнительно были рассчитаны целевые переменные `pIC50`, `pCC50` и `log10_SI` в логарифмической шкале. Такое преобразование применялось для уменьшения асимметрии распределений, ослабления влияния экстремальных значений и приведения целевых показателей к форме, более удобной для статистического анализа и регрессионного моделирования.

Следующим этапом была генерация новых производных признаков на основе исходных дескрипторов. Этот шаг позволил расширить описание молекулярных свойств соединений и включить в анализ дополнительные характеристики, которые могли содержать полезную информацию о биологической активности, цитотоксичности и селективности. Таким образом, формировалось более полное признаковое представление объектов исследования.

После расширения исходного признакового пространства был проведен отбор дескрипторов. На первом этапе оценивалась статистическая связь каждого числового признака с таргетами `pIC50`, `pCC50` и `log10_SI`, затем применялась коррекция множественных проверок для снижения вероятности ложноположительных результатов. На заключительном этапе удалялись сильно коррелирующие между собой признаки, что позволило уменьшить избыточность, снизить мультиколлинеарность и сформировать компактный, устойчивый и интерпретируемый набор дескрипторов для дальнейшего обучения моделей.

In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Загрузка исходных данных

In [3]:
train = pd.read_csv('/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Модель_tabpfn/train.csv')
train.head(5)

,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,102.414420,95.757483,0.935000,5.466584,5.466584,0.719259,0.719259,0.681165,18.307692,...,1,0,0,0,0,0,0,0,0,0
1,1,0.044333,8.401080,189.500000,11.492712,11.492712,0.012350,-3.798024,0.769122,27.652174,...,0,1,0,0,0,0,0,0,0,0
2,2,4.437964,50.085589,11.285714,5.366084,5.366084,0.522930,0.522930,0.612606,24.608696,...,0,0,0,0,0,0,0,0,0,0
3,3,6.827881,682.788051,100.000000,13.317130,13.317130,0.020658,-4.829339,0.345823,12.400000,...,0,0,1,0,0,0,0,0,0,0
4,4,2.003253,70.001455,34.943894,6.320833,6.320833,0.300347,0.300347,0.562066,60.272727,...,0,0,0,0,0,0,0,0,0,0


In [4]:
test = pd.read_csv('/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/Модель_tabpfn/test.csv')
test.head(5)

,index,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,13.761882,13.761882,0.121946,-0.962625,0.770057,30.580645,450.541,432.397,450.070799,...,1,0,0,0,0,0,0,1,0,0
1,1,13.224489,13.224489,0.066132,-1.801871,0.278628,25.687500,448.380,428.220,448.100561,...,0,0,0,0,0,0,0,0,0,0
2,2,6.191528,6.191528,0.445278,0.445278,0.657472,55.384615,179.307,158.139,179.167400,...,0,0,0,0,0,0,0,0,0,0
3,3,14.061236,14.061236,0.054870,-6.660336,0.564307,23.464286,410.289,397.185,410.086525,...,0,0,0,0,0,0,0,0,0,0
4,4,12.790378,12.790378,0.320463,-1.642616,0.696213,22.000000,280.279,268.183,280.073559,...,0,0,0,0,0,0,0,0,0,0


In [5]:
print(train.shape)
print(test.shape)

(751, 214)
(250, 211)


In [6]:
train.drop(columns=['index'], inplace=True, errors='ignore')
test.drop(columns=['index'], inplace=True, errors='ignore')

### Константные признаки для удаления

In [7]:
constant_features = ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_N_O', 'fr_SH', 
                     'fr_azide', 'fr_barbitur', 'fr_benzodiazepine', 'fr_diazo', 
                     'fr_dihydropyridine', 'fr_isocyan', 'fr_isothiocyan', 
                     'fr_lactam', 'fr_nitroso', 'fr_phos_acid', 
                     'fr_phos_ester', 'fr_prisulfonamd', 'fr_thiocyan']

In [8]:
train.isna().sum().sort_values(ascending=False).head(15)

BCUT2D_MRHI            2
BCUT2D_MWHI            2
BCUT2D_CHGHI           2
MinAbsPartialCharge    2
MaxAbsPartialCharge    2
MinPartialCharge       2
MaxPartialCharge       2
BCUT2D_CHGLO           2
BCUT2D_LOGPHI          2
BCUT2D_LOGPLOW         2
BCUT2D_MWLOW           2
BCUT2D_MRLOW           2
fr_N_O                 0
fr_NH2                 0
fr_NH1                 0
dtype: int64

In [9]:
test.isna().sum().sort_values(ascending=False).head(15)

BCUT2D_MWHI            1
MaxPartialCharge       1
BCUT2D_MWLOW           1
BCUT2D_LOGPHI          1
BCUT2D_LOGPLOW         1
BCUT2D_MRHI            1
BCUT2D_MRLOW           1
MinAbsPartialCharge    1
MaxAbsPartialCharge    1
MinPartialCharge       1
BCUT2D_CHGHI           1
BCUT2D_CHGLO           1
fr_Ndealkylation2      0
fr_Ndealkylation1      0
fr_N_O                 0
dtype: int64

## Cоздание  новых признаков

## Описание сконструированных признаков

### Блок 1. Физико-химические ратио (20 признаков)

| Признак | Определение |
|---|---|
| `LogP_per_MW` | **LogP на единицу молярной массы** — нормированная липофильность. Показывает, насколько молекула гидрофобна относительно своего размера. Используется для оценки пассивной диффузии через мембраны. |
| `LogP_per_TPSA` | **LogP / TPSA** — соотношение гидрофобности к полярной поверхности. Высокое значение → молекула липофильна и малополярна → хорошее всасывание, но риск токсичности. |
| `MolMR_per_MW` | **Молярная рефракция / молярная масса** — нормированная поляризуемость. MR отражает способность молекулы к дисперсионным взаимодействиям с белками-мишенями. |
| `TPSA_per_MW` | **TPSA / MW** — плотность полярной поверхности. Нормирует полярность на размер молекулы. Коррелирует с биодоступностью при пероральном введении. |
| `TPSA_per_HA` | **TPSA на тяжёлый атом** — средний полярный вклад каждого нелёгкого атома. Отражает локальную плотность полярных групп в структуре. |
| `HBD_HBA_ratio` | **Доноры / акцепторы водородной связи** — соотношение HBD к HBA. Характеризует баланс между донорной и акцепторной водородно-связывающей способностью молекулы. |
| `HBD_per_MW` | **Число доноров H-связи / MW** — нормированная способность отдавать протон. Связана с растворимостью и биодоступностью. |
| `NHOH_per_HA` | **Число OH/NH групп на тяжёлый атом** — плотность гидроксильных и аминогрупп. Отражает способность к специфическим взаимодействиям с активным центром белка. |
| `Aromatic_ratio` | **Ароматических колец / всего колец** — доля ароматических циклов среди всех. Высокие значения указывают на преобладание π-систем, важных для стэкинг-взаимодействий. |
| `Arom_per_HA` | **Ароматических колец / тяжёлых атомов** — нормированная ароматичность структуры. |
| `Rotatable_per_HA` | **Вращаемые связи / тяжёлые атомы** — гибкость цепи относительно размера молекулы. Высокие значения коррелируют с энтропийными потерями при связывании. |
| `Saturation` | **Насыщенных колец / все кольца** — степень насыщенности циклической системы. Насыщенные кольца (sp³) улучшают растворимость и снижают плоскостность молекулы. |
| `Ring_per_RotBond` | **Число колец / вращаемые связи** — соотношение жёсткости (кольца) к гибкости (ротабельные связи). |
| `LogP_x_TPSA` | **LogP × TPSA** — перекрёстное взаимодействие липофильности и полярности. Нелинейный эффект: очень гидрофобные и очень полярные молекулы ведут себя иначе, чем каждый фактор по отдельности. |
| `NumRings_x_LogP` | **Число колец × LogP** — вклад цикличности в гидрофобность. |
| `FractionCSP3_x_MW` | **Доля sp³-углеродов × MW** — насыщенность структуры, взвешенная по размеру. Высокое значение — объёмная насыщенная молекула с хорошей трёхмерностью. |
| `Chi0_LogP` | **Топологический индекс χ₀ × LogP** — произведение молекулярной связности нулевого порядка на липофильность. Отражает совместный вклад размера и гидрофобности. |
| `Chi1v_LogP` | **χ₁ᵥ × LogP** — валентный индекс связности первого порядка, взвешенный на LogP. |
| `MaxCharge_TPSA` | **Максимальный частичный заряд × TPSA** — взаимодействие электростатики и полярности. |
| `ChargeDiff` | **MaxPartialCharge − MinPartialCharge** — диапазон частичных зарядов. Отражает внутримолекулярную поляризацию и способность к электростатическим взаимодействиям. |

---

## Блок 2. Бинарные агрегаты и взаимодействия (20 признаков)

| Признак | Определение |
|---|---|
| `Heteroatom_groups` | **Сумма тиофеновых, тиазольных, имидазольных и пиридиновых фрагментов** — общее число биоактивных гетероциклов, часто присутствующих в антивирусных и антибактериальных агентах. |
| `Acidic_groups` | **Сумма COO, COO2, Al_COO групп** — число карбоксильных кислотных фрагментов. Влияет на ионизацию, растворимость при физиологическом pH и взаимодействие с катионными сайтами белков. |
| `NH_groups` | **Ar_NH + Nhpyrrole + priamide** — суммарное число NH-содержащих ароматических и амидных групп, участвующих в донорных H-связях. |
| `Sulfur_groups` | **C_S + sulfonamd + sulfone** — число серосодержащих функциональных групп. Сера участвует в координационных взаимодействиях с металлами активного центра и в ковалентном ингибировании. |
| `Nitro_groups` | **nitro + nitro_arom + nitro_arom_nonortho** — суммарное число нитрогрупп. Нитросоединения — потенциальные биоредуктивные пролекарства и электрофильные агенты. |
| `COO_x_thiazole` | **fr_COO × fr_thiazole** — совместное присутствие карбоксильной и тиазольной групп. Часто встречается в β-лактамных антибиотиках и антивирусных соединениях. |
| `Imidazole_x_NH` | **fr_imidazole × fr_Ar_NH** — сочетание имидазола с ароматическим NH. Имидазол — ключевой фармакофор во многих противогрибковых и антигистаминных препаратах. |
| `Thiophene_x_COO` | **fr_thiophene × fr_COO** — совместное присутствие тиофена и карбоксила. Характерно для тиофенкарбоксильных кислот с биологической активностью. |
| `Nitro_x_Arom` | **fr_nitro × NumAromaticRings** — произведение числа нитрогрупп на ароматичность. Отражает наличие нитроароматических систем. |
| `Total_fragments` | **Сумма 10 ключевых бинарных фрагментов** (тиофен, COO, тиазол, имидазол, C_S, Nhpyrrole, Ar_NH, imide, пиридин, нитро). Общая «фармакофорная насыщенность» молекулы. |
| `Fragment_density` | **Total_fragments / HeavyAtomCount** — плотность ключевых фармакофорных фрагментов. Нормирует фармакофорное разнообразие на размер молекулы. |
| `Chi2n_x_LogP` | **χ₂ⁿ × LogP** — индекс связности второго порядка, взвешенный на гидрофобность. |
| `VSA_EState4_x_MW` | **VSA_EState4 / MW** — площадь поверхности по EState4, нормированная на молярную массу. |
| `PEOE_VSA7_x_TPSA` | **PEOE_VSA7 × TPSA** — площадь поверхности частичного выравнивания зарядов (зона 7) × полярная поверхность. Характеризует совместный вклад зарядовой и полярной поверхности. |
| `LogP_TPSA_prod` | **LogP × TPSA** — дубль `LogP_x_TPSA` (удаляется при чистке корреляций = 1). |
| `Chi1v_Chi4v` | **χ₁ᵥ × χ₄ᵥ** — произведение валентных индексов связности 1-го и 4-го порядков. Отражает совместный вклад локальной и дальней молекулярной связности. |
| `MolMR_x_LogP` | **MolMR × LogP** — поляризуемость × гидрофобность. Оба параметра связаны с дисперсионными взаимодействиями при связывании с белком. |
| `HBA_HBD_sum` | **NumHAcceptors + NumHDonors** — суммарная водородно-связывающая способность. Один из критериев правила Липинского (≤ 10 HBA + 5 HBD для биодоступности). |

---

## Блок 3. Нелинейные преобразования (11 признаков)

| Признак | Определение |
|---|---|
| `VSA_EState4_sq` | **VSA_EState4²** — квадрат площади поверхности по EState4. Нелинейно усиливает вклад молекул с крупными электротопологическими зонами. |
| `VSA_EState4_sq2` | Технический синоним `VSA_EState4_sq` (удаляется при чистке). |
| `Chi2n_sq` | **χ₂ⁿ²** — квадрат топологического индекса связности 2-го порядка. Нелинейная характеристика разветвлённости структуры. |
| `Chi2v_sqrt` | **√Chi2v** (clip ≥ 0) — квадратный корень из валентного индекса связности. Сглаживает распределение для крупных молекул. |
| `MolMR_log` | **log(1 + MolMR)** — логарифм молярной рефракции. Стабилизирует правостороннее распределение MR и приближает его к нормальному. |
| `TPSA_log` | **log(1 + TPSA)** — логарифм полярной поверхности. Улучшает линейность связи TPSA с биодоступностью и всасываемостью. |
| `VSA_mean` | **Среднее по всем VSA-признакам** — средняя площадь поверхности, взвешенная по различным физико-химическим зонам. |
| `VSA_std` | **СКО по VSA-признакам** — разброс между зонами поверхности. Высокое значение → неравномерное распределение зарядовых/полярных зон. |
| `Chi_mean` | **Среднее по всем χ-индексам** — среднее топологическое разнообразие структуры. |
| `Chi_sum` | **Сумма χ-индексов** — суммарная топологическая связность (удаляется при чистке, коллинеарен с `Chi_mean`). |
| `Chi_max` | **Максимальный χ-индекс** — наибольший индекс связности. Отражает доминирующую топологическую характеристику молекулы. |

---

## Блок 4. Полиномиальные взаимодействия (10 признаков)

Все признаки этого блока — **попарные произведения** топ-5 дескрипторов с наибольшей предиктивной силой: `VSA_EState4`, `Chi2n`, `Chi2v`, `PEOE_VSA7`, `Chi4v`.

| Признак | Определение |
|---|---|
| `VSA_EState4_Chi2n` | VSA_EState4 × χ₂ⁿ — совместный вклад электротопологической поверхности и разветвлённости 2-го порядка. |
| `VSA_EState4_Chi2v` | VSA_EState4 × χ₂ᵥ (удаляется при чистке, коллинеарен с Chi2n). |
| `VSA_EState4_PEOE_VSA7` | VSA_EState4 × PEOE_VSA7 (удаляется при чистке). |
| `VSA_EState4_Chi4v` | VSA_EState4 × χ₄ᵥ (удаляется при чистке). |
| `Chi2n_Chi2v` | χ₂ⁿ × χ₂ᵥ — взаимодействие топологического и валентного индексов 2-го порядка (удаляется при чистке). |
| `Chi2n_PEOE_VSA7` | χ₂ⁿ × PEOE_VSA7 — разветвлённость × зарядовая поверхность. |
| `Chi2n_Chi4v` | χ₂ⁿ × χ₄ᵥ — взаимодействие разветвлённости на малых и дальних расстояниях. |
| `Chi2v_PEOE_VSA7` | χ₂ᵥ × PEOE_VSA7 — валентная связность × электростатическая поверхность. |
| `Chi2v_Chi4v` | χ₂ᵥ × χ₄ᵥ — совместный вклад валентной связности 2-го и 4-го порядков. |
| `PEOE_VSA7_Chi4v` | PEOE_VSA7 × χ₄ᵥ — электростатическая поверхность × дальняя топологическая связность. |

---

## Блок 5. Агрегаты и комплексные индексы (35 признаков)

### 5.1 EState агрегаты

| Признак | Определение |
|---|---|
| `EState_mean` | **Среднее по EState_VSA-зонам** — средний электротопологический вклад зон поверхности. EState (Electronic State) отражает доступность электронов в каждой части молекулы. |
| `EState_std` | **СКО по EState_VSA-зонам** — неравномерность электронного распределения по поверхности. |
| `EState_max` | **Максимальная EState_VSA-зона** — доминирующая электронно-богатая область поверхности. |
| `EState_min` | **Минимальная EState_VSA-зона** — электронно-бедная область, потенциальный акцептор. |

### 5.2 Нормированные ратио и произведения

| Признак | Определение |
|---|---|
| `VSA_to_MolMR` | **VSA_mean / MolMR** — соотношение средней поверхности к молярной рефракции. Характеризует, насколько поверхность «упакована» относительно поляризуемости. |
| `Chi_to_RingCount` | **Chi_mean / RingCount** — средняя связность на кольцо. |
| `Hetero_to_Carbon` | **NumHeteroatoms / (C-атомы + 1)** — гетероатомный индекс. Отражает долю N, O, S, галогенов относительно углеродного скелета. |
| `Polar_Surface_Density` | **TPSA / HeavyAtomCount** — полярная поверхность на атом. Дубль `TPSA_per_HA` (удаляется при чистке). |
| `LogP_efficiency` | **LogP / HeavyAtomCount** — липофильная эффективность (LipE). Широко используется в medicinal chemistry: высокая активность при низком LogP. |
| `H_bond_capacity` | **(HBD + HBA) / MW** — суммарная водородно-связывающая ёмкость, нормированная на массу. |
| `Ring_to_Atom` | **RingCount / HeavyAtomCount** — доля атомов, включённых в циклические системы. |
| `Arom_to_Sat` | **NumAromaticRings / (NumSaturatedRings + 1)** — отношение ароматических колец к насыщенным. Высокие значения → плоская ароматическая молекула. |
| `MW_x_LogP` | **MW × LogP** — размер × гидрофобность. Крупные гидрофобные молекулы часто имеют проблемы с биодоступностью. |
| `TPSA_x_HBD` | **TPSA × NumHDonors** — полярная поверхность, усиленная донорными группами. |
| `MolMR_x_TPSA` | **MolMR × TPSA** — поляризуемость × полярность. |
| `Chi0_x_RingCount` | **χ₀ × RingCount** — молекулярная связность × цикличность. |
| `Kappa1_x_Kappa2` | **κ₁ × κ₂** — произведение индексов молекулярной формы Холла-Кира 1-го и 2-го порядков. Характеризует двухуровневую форму молекулы. |
| `BalabanJ_x_LogP` | **BalabanJ × LogP** — индекс Балабана (топологическая компактность) × липофильность. Отражает компактность гидрофобной молекулы. |

### 5.3 Молекулярные индексы сложности

| Признак | Определение |
|---|---|
| `Molecular_Complexity` | **BertzCT / HeavyAtomCount** — структурная сложность Бертца на атом. CT (Complexity Total) — информационно-теоретическая мера разнообразия связей. |
| `Structural_Diversity` | **Chi_max − Chi_mean** — разброс топологических индексов. Отражает неоднородность связности в молекуле. |
| `Heteroatom_Richness` | **NumHeteroatoms / MW** — обогащённость гетероатомами относительно массы. |
| `Charge_Asymmetry` | **|MaxPartialCharge + MinPartialCharge|** — суммарная поляризация молекулы. Высокое значение → выраженный диполь. |
| `VSA_Diversity` | **VSA_std / |VSA_mean|** — коэффициент вариации VSA-зон. Отражает неравномерность распределения поверхностных свойств. |
| `Chi_Complexity` | **Chi_max / |Chi_mean|** — отношение максимальной к средней связности. |
| `Fragment_Complexity` | **Total_fragments / (RingCount + 1)** — фармакофорная насыщенность на кольцо. |
| `Lipinski_Score` | **4 − Lipinski_Violations** — число выполненных правил Липинского из 4 (MW ≤ 500, LogP ≤ 5, HBD ≤ 5, HBA ≤ 10). Прокси биодоступности при пероральном введении. |
| `Rotatable_Flexibility` | **NumRotatableBonds / (RingCount + 1)** — гибкость цепи относительно циклической жёсткости. Предсказывает энтропийные потери при связывании с мишенью. |
| `Aromatic_Density` | **NumAromaticRings / MW** — ароматичность на единицу массы. |

### 5.4 Квадраты ключевых признаков

| Признак | Определение |
|---|---|
| `PEOE_VSA7_sq` | **PEOE_VSA7²** — нелинейный вклад 7-й зоны электростатической поверхности. |
| `Chi4v_sq` | **χ₄ᵥ²** — квадрат валентного индекса 4-го порядка. Нелинейно усиливает вклад дальней топологической связности. |
| `MolLogP_sq` | **LogP²** — квадратичный член для LogP. Улучшает аппроксимацию U-образных зависимостей активность–LogP (оптимум ≈ 2–3). |
| `TPSA_sq` | **TPSA²** — квадрат полярной поверхности. Нелинейно усиливает вклад высокополярных молекул. |

### 5.5 Взаимодействия VSA_EState4

| Признак | Определение |
|---|---|
| `VSA_EState4_x_PEOE7` | **VSA_EState4 × PEOE_VSA7** — электротопологическая × электростатическая поверхность (удаляется при чистке). |
| `VSA_EState4_x_Chi4v` | **VSA_EState4 × χ₄ᵥ** — электротопологическая поверхность × дальняя связность (удаляется при чистке). |

---

## Восстановленные признаки из `add_restored_features`

| Признак | Определение |
|---|---|
| `Heteroatom_Count` | **NOCount + NHOHCount + сера-содержащие фрагменты** — полное число гетероатомов (N, O, S). Обобщённый индикатор полярности и способности к H-связям и координации. |
| `Lipinski_Violations` | **Число нарушений правил Липинского** (из 4). 0–1 нарушение → хорошая пероральная биодоступность; ≥ 2 → вероятные проблемы. |
| `LogD_pH7.4_approx` | **Приближение LogD при pH 7.4** — эффективная липофильность при физиологическом pH (≈ LogP для незаряженных молекул). |
| `Polar_to_Nonpolar_Ratio` | **TPSA / (MW − TPSA)** — отношение полярной площади к неполярной. Характеризует баланс гидрофильных и гидрофобных поверхностей. |
| `Aromatic_Heteroatom_Ratio` | **NumAromaticHeterocycles / (NumAromaticCarbocycles + NumAromaticHeterocycles)** — доля ароматических гетероциклов среди всех ароматических колец. |
| `Caco2_Permeability_approx` | **−5.4 − 0.01·TPSA + 0.6·LogP** — модель Palm et al. (1997) для оценки кишечной проницаемости через клетки Caco-2. Прокси всасывания в ЖКТ. |
| `Aqueous_Solubility_approx` | **0.5 − LogP − 0.01·MW** — модель Delaney (2004, ESOL) для оценки растворимости в воде. Предсказывает log(растворимость в моль/л). |


In [10]:
def add_restored_features(df):

    # 1. Heteroatom_Count
    sulfur_cols = ['fr_C_S', 'fr_thiophene', 'fr_sulfonamd', 'fr_sulfone', 'fr_SH']
    sulfur_present = [c for c in sulfur_cols if c in df.columns]
    df['Heteroatom_Count'] = df['NOCount'] + df['NHOHCount']
    if sulfur_present:
        df['Heteroatom_Count'] += df[sulfur_present].sum(axis=1)

    # 2. Lipinski_Violations
    df['Lipinski_Violations'] = (
        (df['MolWt'] > 500).astype(int) +
        (df['MolLogP'] > 5  ).astype(int) +
        (df['NumHDonors'] > 5  ).astype(int) +
        (df['NumHAcceptors'] > 10 ).astype(int)
    )

    # 3. LogD_pH7.4_approx
    df['LogD_pH7.4_approx'] = df['MolLogP'].copy()

    # 4. Polar_to_Nonpolar_Ratio
    nonpolar = (df['MolWt'] - df['TPSA']).clip(lower=1e-6)
    df['Polar_to_Nonpolar_Ratio'] = df['TPSA'] / nonpolar

    # 5. Aromatic_Heteroatom_Ratio
    total_arom = df['NumAromaticCarbocycles'] + df['NumAromaticHeterocycles']
    df['Aromatic_Heteroatom_Ratio'] = np.where(
        total_arom > 0,
        df['NumAromaticHeterocycles'] / total_arom, 0.0
    )

    # 6. Caco2_Permeability_approx  (Palm et al. 1997)
    df['Caco2_Permeability_approx'] = -5.4 - 0.01*df['TPSA'] + 0.6*df['MolLogP']

    # 7. Aqueous_Solubility_approx  (Delaney 2004)
    df['Aqueous_Solubility_approx'] = 0.5 - df['MolLogP'] - 0.01*df['MolWt']

    return df

In [11]:
train = add_restored_features(train).copy()
test  = add_restored_features(test).copy()

In [12]:
def create_all_features(df):
    """Создание 93 новых признаков из исходных RDKit-дескрипторов."""

    # === 1. Физико-химические ратио (20) ===
    df['LogP_per_MW'] = df['MolLogP'] / (df['MolWt'] + 1)
    df['LogP_per_TPSA'] = df['MolLogP'] / (df['TPSA'] + 1)
    df['MolMR_per_MW'] = df['MolMR'] / (df['MolWt'] + 1)
    df['TPSA_per_MW'] = df['TPSA'] / (df['MolWt'] + 1)
    df['TPSA_per_HA'] = df['TPSA'] / (df['HeavyAtomCount'] + 1)
    df['HBD_HBA_ratio'] = df['NumHDonors'] / (df['NumHAcceptors'] + 1)
    df['HBD_per_MW'] = df['NumHDonors'] / (df['MolWt'] + 1)
    df['NHOH_per_HA'] = df['NHOHCount'] / (df['HeavyAtomCount'] + 1)
    df['Aromatic_ratio'] = df['NumAromaticRings'] / (df['RingCount'] + 1)
    df['Arom_per_HA'] = df['NumAromaticRings'] / (df['HeavyAtomCount'] + 1)
    df['Rotatable_per_HA'] = df['NumRotatableBonds'] / (df['HeavyAtomCount'] + 1)
    df['Saturation'] = df['NumSaturatedRings'] / (df['RingCount'] + 1)
    df['Ring_per_RotBond'] = df['RingCount'] / (df['NumRotatableBonds'] + 1)
    df['LogP_x_TPSA'] = df['MolLogP'] * df['TPSA']
    df['NumRings_x_LogP'] = df['RingCount'] * df['MolLogP']
    df['FractionCSP3_x_MW'] = df['FractionCSP3'] * df['MolWt']
    df['Chi0_LogP'] = df['Chi0'] * df['MolLogP']
    df['Chi1v_LogP'] = df['Chi1v'] * df['MolLogP']
    df['MaxCharge_TPSA'] = df['MaxPartialCharge'] * df['TPSA']
    df['ChargeDiff'] = df['MaxPartialCharge'] - df['MinPartialCharge']

    # === 2. Бинарные агрегаты (20, было 18 — добавлено 2) ===
    df['Heteroatom_groups'] = df['fr_thiophene'] + df['fr_thiazole'] + df['fr_imidazole'] + df['fr_pyridine']
    df['Acidic_groups'] = df['fr_COO'] + df['fr_COO2'] + df['fr_Al_COO']
    df['NH_groups'] = df['fr_Ar_NH'] + df['fr_Nhpyrrole'] + df['fr_priamide']
    df['Sulfur_groups'] = df['fr_C_S'] + df['fr_sulfonamd'] + df['fr_sulfone']
    df['Nitro_groups'] = df['fr_nitro'] + df['fr_nitro_arom'] + df['fr_nitro_arom_nonortho']
    df['COO_x_thiazole'] = df['fr_COO'] * df['fr_thiazole']
    df['Imidazole_x_NH'] = df['fr_imidazole'] * df['fr_Ar_NH']
    df['Thiophene_x_COO'] = df['fr_thiophene'] * df['fr_COO']
    df['Nitro_x_Arom'] = df['fr_nitro'] * df['NumAromaticRings']
    binary_cols = ['fr_thiophene','fr_COO','fr_thiazole','fr_imidazole','fr_C_S',
                   'fr_Nhpyrrole','fr_Ar_NH','fr_imide','fr_pyridine','fr_nitro']
    df['Total_fragments'] = df[binary_cols].sum(axis=1)
    df['Fragment_density'] = df['Total_fragments'] / (df['HeavyAtomCount'] + 1)
    df['Chi2n_x_LogP'] = df['Chi2n'] * df['MolLogP']
    df['VSA_EState4_x_MW'] = df['VSA_EState4'] / (df['MolWt'] + 1)
    df['PEOE_VSA7_x_TPSA'] = df['PEOE_VSA7'] * df['TPSA']
    df['LogP_TPSA_prod'] = df['MolLogP'] * df['TPSA']
    df['Chi1v_Chi4v'] = df['Chi1v'] * df['Chi4v']
    df['MolMR_x_LogP'] = df['MolMR'] * df['MolLogP']
    df['HBA_HBD_sum'] = df['NumHAcceptors'] + df['NumHDonors']

    # === 3. Нелинейные (11, было 10 — добавлен VSA_EState4_sq2) ===
    df['VSA_EState4_sq'] = df['VSA_EState4'] ** 2
    df['VSA_EState4_sq2'] = df['VSA_EState4'] ** 2   # синоним — для совместимости с ожидаемым списком
    df['Chi2n_sq'] = df['Chi2n'] ** 2
    df['Chi2v_sqrt'] = np.sqrt(df['Chi2v'].clip(lower=0))
    df['MolMR_log'] = np.log1p(df['MolMR'])
    df['TPSA_log'] = np.log1p(df['TPSA'])

    VSA_cols = [c for c in df.columns if 'VSA' in c and c not in ['VSA_mean', 'VSA_std']]
    df['VSA_mean'] = df[VSA_cols].mean(axis=1)
    df['VSA_std'] = df[VSA_cols].std(axis=1)

    Chi_cols = [c for c in df.columns if c.startswith('Chi') and c not in ['Chi_mean', 'Chi_sum', 'Chi_max']]
    df['Chi_mean'] = df[Chi_cols].mean(axis=1)
    df['Chi_sum'] = df[Chi_cols].sum(axis=1)
    df['Chi_max'] = df[Chi_cols].max(axis=1)

    # === 4. Полиномиальные взаимодействия (10) ===
    top5 = ['VSA_EState4', 'Chi2n', 'Chi2v', 'PEOE_VSA7', 'Chi4v']
    poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
    poly_feats = poly.fit_transform(df[top5])
    # interaction_only=True → ровно C(5,2)=10 столбцов (индексы 0–9)
    poly_names = [
        'VSA_EState4_Chi2n', 'VSA_EState4_Chi2v', 'VSA_EState4_PEOE_VSA7',
        'VSA_EState4_Chi4v', 'Chi2n_Chi2v', 'Chi2n_PEOE_VSA7',
        'Chi2n_Chi4v', 'Chi2v_PEOE_VSA7', 'Chi2v_Chi4v', 'PEOE_VSA7_Chi4v'
    ]
    for i, name in enumerate(poly_names):
        df[name] = poly_feats[:, i]   # ← исправлено: было i+5, должно быть i

    # === 5. Агрегаты и комплексные (35) ===
    EState_cols = [c for c in df.columns if c.startswith('EState_VSA')]
    df['EState_mean'] = df[EState_cols].mean(axis=1)
    df['EState_std'] = df[EState_cols].std(axis=1)
    df['EState_max'] = df[EState_cols].max(axis=1)
    df['EState_min'] = df[EState_cols].min(axis=1)

    df['VSA_to_MolMR'] = df['VSA_mean'] / (df['MolMR'] + 1)
    df['Chi_to_RingCount'] = df['Chi_mean'] / (df['RingCount'] + 1)
    df['Hetero_to_Carbon'] = df['NumHeteroatoms'] / (df['HeavyAtomCount'] - df['NumHeteroatoms'] + 1)
    df['Polar_Surface_Density'] = df['TPSA'] / (df['HeavyAtomCount'] + 1)
    df['LogP_efficiency'] = df['MolLogP'] / (df['HeavyAtomCount'] + 1)
    df['H_bond_capacity'] = (df['NumHDonors'] + df['NumHAcceptors']) / (df['MolWt'] + 1)
    df['Ring_to_Atom'] = df['RingCount'] / (df['HeavyAtomCount'] + 1)
    df['Arom_to_Sat'] = df['NumAromaticRings'] / (df['NumSaturatedRings'] + 1)
    df['MW_x_LogP'] = df['MolWt'] * df['MolLogP']
    df['TPSA_x_HBD'] = df['TPSA'] * df['NumHDonors']
    df['MolMR_x_TPSA'] = df['MolMR'] * df['TPSA']
    df['Chi0_x_RingCount'] = df['Chi0'] * df['RingCount']
    df['Kappa1_x_Kappa2'] = df['Kappa1'] * df['Kappa2']
    df['BalabanJ_x_LogP'] = df['BalabanJ'] * df['MolLogP']
    df['Molecular_Complexity'] = df['BertzCT'] / (df['HeavyAtomCount'] + 1)
    df['Structural_Diversity'] = df['Chi_max'] - df['Chi_mean']
    df['Heteroatom_Richness'] = df['NumHeteroatoms'] / (df['MolWt'] + 1)
    df['Charge_Asymmetry'] = (df['MaxPartialCharge'] + df['MinPartialCharge']).abs()
    df['VSA_Diversity'] = df['VSA_std'] / (df['VSA_mean'].abs() + 1)
    df['Chi_Complexity'] = df['Chi_max'] / (df['Chi_mean'].abs() + 1)
    df['Fragment_Complexity'] = df['Total_fragments'] / (df['RingCount'] + 1)
    df['Lipinski_Score'] = 4 - (
                                (df['MolWt'] > 500).astype(int) +
                                (df['MolLogP'] > 5  ).astype(int) +
                                (df['NumHDonors'] > 5  ).astype(int) +
                                (df['NumHAcceptors'] > 10 ).astype(int)
                                  )
    df['Rotatable_Flexibility'] = df['NumRotatableBonds'] / (df['RingCount'] + 1)
    df['Aromatic_Density'] = df['NumAromaticRings'] / (df['MolWt'] + 1)
    df['PEOE_VSA7_sq'] = df['PEOE_VSA7'] ** 2
    df['Chi4v_sq'] = df['Chi4v'] ** 2
    df['MolLogP_sq'] = df['MolLogP'] ** 2
    df['TPSA_sq'] = df['TPSA'] ** 2
    df['VSA_EState4_x_PEOE7'] = df['VSA_EState4'] * df['PEOE_VSA7']
    df['VSA_EState4_x_Chi4v'] = df['VSA_EState4'] * df['Chi4v']

    return df

In [13]:
# Применение
train_enriched = create_all_features(train.copy())
test_enriched  = create_all_features(test.copy())

print(f"До: {train.shape[1]} столбцов")
print(f"После: {train_enriched.shape[1]} столбцов")
print(f"Добавлено: {train_enriched.shape[1] - train.shape[1]} новых признаков")

До: 220 столбцов
После: 313 столбцов
Добавлено: 93 новых признаков


In [14]:
train_enriched = train_enriched.drop(columns=constant_features, errors='ignore')
test_enriched  = test_enriched.drop(columns=constant_features, errors='ignore')

In [15]:
train_enriched.shape, test_enriched.shape

((751, 295), (250, 292))

### Логарифмическоe преобразование таргитов

Мы применяем логарифмическое преобразование к `IC50`, `CC50` и `SI`, чтобы уменьшить асимметрию распределений, снизить влияние выбросов и привести значения к более стабильной шкале для обучения модели. Кроме того, `pIC50` и `pCC50` удобны для интерпретации: большие значения соответствуют более сильной активности и более высокой цитотоксичности соответственно, а `log10_SI` упрощает моделирование индекса селективности.

In [16]:
train_enriched['pIC50'] = -np.log10(train_enriched['IC50, mM'] / 1000.0)
train_enriched['pCC50'] = -np.log10(train_enriched['CC50, mM'] / 1000.0)
train_enriched['log10_SI'] = np.log10(train_enriched['SI'])

### Удаление признаков с корреляцией равной 1

В данном блоке  выполняется поиск пар числовых признаков с корреляцией, равной 1 с высокой точностью. Это нужно для выявления дублирующих или почти идентичных признаков; удаление одного признака из такой пары выполняется отдельным шагом после анализа найденных совпадений.

In [17]:
# Таргеты
targets = ['pIC50', 'pCC50', 'log10_SI']
existing_targets = [t for t in targets if t in train_enriched.columns]
print(f"Таргеты: {existing_targets}")

# Берём только числовые признаки (без таргетов и служебных)
service_cols = ['index', 'IC50, mM', 'CC50, mM', 'SI'] + existing_targets
feature_cols = [c for c in train_enriched.select_dtypes(include=[np.number]).columns if c not in service_cols]
print(f"Признаков для анализа: {len(feature_cols)}")

feat_df = train_enriched[feature_cols].copy()

# Матрица корреляций между признаками
corr_matrix = feat_df.corr(method='pearson').abs()

# Находим все пары с корреляцией == 1.0 (с точностью до 1e-10)
pairs_to_check = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        val = corr_matrix.iloc[i, j]
        if val >= 0.9999999:
            pairs_to_check.append((cols[i], cols[j], val))

print(f"\nПар с корреляцией == 1.0: {len(pairs_to_check)}")
for a, b, v in pairs_to_check:
    print(f"  {a}  <-->  {b}  ({v:.10f})")

Таргеты: ['pIC50', 'pCC50', 'log10_SI']
Признаков для анализа: 292

Пар с корреляцией == 1.0: 19
  MaxAbsEStateIndex  <-->  MaxEStateIndex  (1.0000000000)
  Chi2n  <-->  VSA_EState4_Chi2v  (1.0000000000)
  Chi2v  <-->  VSA_EState4_PEOE_VSA7  (1.0000000000)
  Chi4v  <-->  Chi2n_Chi2v  (1.0000000000)
  PEOE_VSA7  <-->  VSA_EState4_Chi4v  (1.0000000000)
  VSA_EState4  <-->  VSA_EState4_Chi2n  (1.0000000000)
  NumAromaticCarbocycles  <-->  fr_benzene  (1.0000000000)
  MolLogP  <-->  LogD_pH7.4_approx  (1.0000000000)
  fr_Ar_NH  <-->  fr_Nhpyrrole  (1.0000000000)
  fr_COO  <-->  fr_COO2  (1.0000000000)
  fr_nitro_arom  <-->  fr_nitro_arom_nonortho  (1.0000000000)
  fr_phenol  <-->  fr_phenol_noOrthoHbond  (1.0000000000)
  Lipinski_Violations  <-->  Lipinski_Score  (1.0000000000)
  TPSA_per_HA  <-->  Polar_Surface_Density  (1.0000000000)
  LogP_x_TPSA  <-->  LogP_TPSA_prod  (1.0000000000)
  VSA_EState4_sq  <-->  VSA_EState4_sq2  (1.0000000000)
  Chi_mean  <-->  Chi_sum  (1.0000000000)
  Chi2

In [18]:
log_targets = ['pIC50', 'pCC50', 'log10_SI']

target_corr = train_enriched[feature_cols].corrwith(train_enriched['pIC50']).abs().rename('pIC50')
for t in ['pCC50', 'log10_SI']:
    target_corr = pd.concat(
        [target_corr, train_enriched[feature_cols].corrwith(train_enriched[t]).abs().rename(t)], axis=1)
target_corr['max_target_corr'] = target_corr[log_targets].max(axis=1)

print("Корреляции с таргетами для пар (оставляем признак с БОЛЬШЕЙ max_target_corr):\n")
to_drop = []
for a, b, v in pairs_to_check:
    corr_a = target_corr.loc[a, 'max_target_corr'] if a in target_corr.index else 0
    corr_b = target_corr.loc[b, 'max_target_corr'] if b in target_corr.index else 0
    keep   = a if corr_a >= corr_b else b
    drop   = b if corr_a >= corr_b else a
    to_drop.append(drop)
    print(f"  {a:40s} corr={corr_a:.4f}")
    print(f"  {b:40s} corr={corr_b:.4f}")
    print(f"  → УДАЛИТЬ: {drop}  |  ОСТАВИТЬ: {keep}\n")

to_drop_unique = list(dict.fromkeys(to_drop))  # убираем дубли, сохраняя порядок
print(f"Итого к удалению: {len(to_drop_unique)} признаков")
print(to_drop_unique)

Корреляции с таргетами для пар (оставляем признак с БОЛЬШЕЙ max_target_corr):

  MaxAbsEStateIndex                        corr=0.1441
  MaxEStateIndex                           corr=0.1441
  → УДАЛИТЬ: MaxEStateIndex  |  ОСТАВИТЬ: MaxAbsEStateIndex

  Chi2n                                    corr=0.1433
  VSA_EState4_Chi2v                        corr=0.1433
  → УДАЛИТЬ: VSA_EState4_Chi2v  |  ОСТАВИТЬ: Chi2n

  Chi2v                                    corr=0.1471
  VSA_EState4_PEOE_VSA7                    corr=0.1471
  → УДАЛИТЬ: VSA_EState4_PEOE_VSA7  |  ОСТАВИТЬ: Chi2v

  Chi4v                                    corr=0.1154
  Chi2n_Chi2v                              corr=0.1154
  → УДАЛИТЬ: Chi2n_Chi2v  |  ОСТАВИТЬ: Chi4v

  PEOE_VSA7                                corr=0.1839
  VSA_EState4_Chi4v                        corr=0.1839
  → УДАЛИТЬ: VSA_EState4_Chi4v  |  ОСТАВИТЬ: PEOE_VSA7

  VSA_EState4                              corr=0.2225
  VSA_EState4_Chi2n                        co

In [19]:
# удаляем
train_enriched = train_enriched.drop(columns=[c for c in to_drop_unique if c in train_enriched.columns])
test_enriched  = test_enriched.drop(columns=[c for c in to_drop_unique if c in test_enriched.columns])

print(f"\ntrain_enriched: {train_enriched.shape}")
print(f"test_enriched:  {test_enriched.shape}")


train_enriched: (751, 279)
test_enriched:  (250, 273)


In [20]:
train = train_enriched.copy()
test = test_enriched.copy()

### Удаление исходных таргитов

In [22]:
train = train.drop(columns=['IC50, mM',  'CC50, mM','SI'], errors='ignore').copy()
train.head(5)

,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,MaxPartialCharge,...,Fragment_Complexity,Rotatable_Flexibility,Aromatic_Density,PEOE_VSA7_sq,Chi4v_sq,MolLogP_sq,TPSA_sq,pIC50,pCC50,log10_SI
0,5.466584,0.719259,0.719259,0.681165,18.307692,195.287,182.183,195.071785,70,0.119177,...,0.00,0.333333,0.005095,147.203238,4.415393,4.714978,155.5009,0.989639,1.018827,-0.029188
1,11.492712,0.012350,-3.798024,0.769122,27.652174,360.907,335.707,360.127441,130,0.237676,...,0.00,1.333333,0.002763,2332.073513,22.467312,8.860148,8541.4564,4.353274,2.075665,2.277609
2,5.366084,0.522930,0.522930,0.612606,24.608696,315.457,286.225,315.219829,126,0.160487,...,0.00,1.400000,0.003160,3691.073968,17.456159,14.655881,929.6401,2.352816,1.300287,1.052529
3,13.317130,0.020658,-4.829339,0.345823,12.400000,439.375,427.279,439.056210,156,0.436923,...,0.75,1.250000,0.006812,1324.829139,9.488829,23.257471,17077.2624,2.165714,0.165714,2.000000
4,6.320833,0.300347,0.300347,0.562066,60.272727,151.253,134.117,151.136100,62,0.016184,...,0.00,0.000000,0.000000,3167.286207,17.452622,3.663013,677.0404,2.698264,1.154893,1.543371


### Замена пропусков в исходных и новых признаков на медиану

In [23]:
train.isna().sum().sort_values(ascending=False).head(15)

MaxCharge_TPSA         2
MaxAbsPartialCharge    2
Charge_Asymmetry       2
BCUT2D_MRHI            2
BCUT2D_LOGPLOW         2
BCUT2D_LOGPHI          2
BCUT2D_CHGLO           2
BCUT2D_CHGHI           2
BCUT2D_MWLOW           2
BCUT2D_MWHI            2
MinAbsPartialCharge    2
BCUT2D_MRLOW           2
MinPartialCharge       2
MaxPartialCharge       2
ChargeDiff             2
dtype: int64

In [24]:
test.isna().sum().sort_values(ascending=False).head(15)

BCUT2D_MRHI            1
MaxAbsPartialCharge    1
BCUT2D_MRLOW           1
BCUT2D_LOGPLOW         1
BCUT2D_LOGPHI          1
BCUT2D_CHGLO           1
BCUT2D_CHGHI           1
BCUT2D_MWLOW           1
BCUT2D_MWHI            1
MinAbsPartialCharge    1
Charge_Asymmetry       1
MinPartialCharge       1
MaxPartialCharge       1
MaxCharge_TPSA         1
ChargeDiff             1
dtype: int64

In [25]:
cols_with_nan = train.columns[train.isna().any()].tolist()
train_mean = train[cols_with_nan].mean()
train = train.fillna(train_mean)
test = test.fillna(train_mean)
print(f"Пропущенные значения в train: {train.isna().sum().sum()}")
print(f"Пропущенные значения в test: {test.isna().sum().sum()}")

Пропущенные значения в train: 0
Пропущенные значения в test: 0


### Отбор информативных признаков для моделей

Данный этап выполняется для формирования единого набора информативных дескрипторов для таргетов `pIC50`, `pCC50` и `log10_SI`. Сначала отбираются признаки, статистически значимо связанные хотя бы с одним таргетом, затем применяется FDR-коррекция для снижения числа ложноположительных находок, а после этого удаляются сильно коррелирующие между собой признаки. В результате получается компактное, устойчивое и менее избыточное признаковое пространство, удобное для дальнейшего построения моделей.

In [26]:
# =========================================================
# 0. НАСТРОЙКИ
# =========================================================
target_cols = ["pIC50", "pCC50", "log10_SI"]
corr_threshold = 0.85
fdr_alpha = 0.05

# =========================================================
# 1. X: ОБЩИЕ ПРИЗНАКИ БЕЗ ТАРГЕТОВ
# =========================================================
X = train.drop(columns=target_cols, errors="ignore").copy()

# оставим только числовые признаки
X = X.select_dtypes(include=[np.number]).copy()

print(f"Исходное количество числовых признаков: {X.shape[1]}")

# =========================================================
# 2. ОТБОР ПО КАЖДОМУ ТАРГЕТУ ОТДЕЛЬНО
#    НО С СОХРАНЕНИЕМ ОБЩЕЙ ТАБЛИЦЫ
# =========================================================
all_results = []

for target_col in target_cols:
    y = train[target_col].copy()

    descriptors = X.columns.tolist()
    p_values = []
    correlations = []

    for col in descriptors:
        mask = X[col].notna() & y.notna()

        if mask.sum() < 3 or X.loc[mask, col].nunique() < 2:
            corr, p_val = 0.0, 1.0
        else:
            corr, p_val = spearmanr(X.loc[mask, col], y.loc[mask])

            if np.isnan(corr):
                corr = 0.0
            if np.isnan(p_val):
                p_val = 1.0

        correlations.append(corr)
        p_values.append(p_val)

    results_df_target = pd.DataFrame({
        "Descriptor": descriptors,
        "Target": target_col,
        "Correlation": correlations,
        "AbsCorrelation": np.abs(correlations),
        "Raw_p_value": p_values
    })

    rejected, q_values, _, _ = multipletests(
        results_df_target["Raw_p_value"],
        alpha=fdr_alpha,
        method="fdr_bh"
    )

    results_df_target["FDR_q_value"] = q_values
    results_df_target["Is_Significant"] = rejected

    all_results.append(results_df_target)

results_long = pd.concat(all_results, axis=0, ignore_index=True)

# =========================================================
# 3. АГРЕГАЦИЯ ПО ПРИЗНАКАМ ДЛЯ 3 ТАРГЕТОВ
#    ЛОГИКА:
#    - оставить признак, если он значим хотя бы для одного таргета
#    - для силы признака взять max(|corr|)
#    - для q-value взять min(q)
#    - посчитать, для скольких таргетов он значим
# =========================================================
agg_results = (
    results_long
    .groupby("Descriptor")
    .agg(
        MaxAbsCorrelation=("AbsCorrelation", "max"),
        MeanAbsCorrelation=("AbsCorrelation", "mean"),
        Min_FDR_q_value=("FDR_q_value", "min"),
        Significant_Targets_Count=("Is_Significant", "sum"),
    )
    .reset_index()
)

agg_results["Keep_By_Union"] = agg_results["Significant_Targets_Count"] > 0

significant_desc = agg_results.loc[
    agg_results["Keep_By_Union"], "Descriptor"
].tolist()

X_fdr = X[significant_desc].copy()

print(f"Осталось после multi-target FDR union: {len(significant_desc)}")

# =========================================================
# 4. УДАЛЕНИЕ МУЛЬТИКОЛЛИНЕАРНОСТИ
#    Если два признака сильно коррелируют между собой,
#    оставляем тот, у которого выше MaxAbsCorrelation
#    по трём таргетам
# =========================================================
corr_matrix = X_fdr.corr(method="spearman").abs()

target_corr_dict = agg_results.set_index("Descriptor")["MaxAbsCorrelation"].to_dict()
sig_count_dict = agg_results.set_index("Descriptor")["Significant_Targets_Count"].to_dict()
qvalue_dict = agg_results.set_index("Descriptor")["Min_FDR_q_value"].to_dict()

to_drop = set()
columns = corr_matrix.columns.tolist()

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        col_A = columns[i]
        col_B = columns[j]

        if col_A in to_drop or col_B in to_drop:
            continue

        if corr_matrix.loc[col_A, col_B] >= corr_threshold:
            score_A = (
                sig_count_dict.get(col_A, 0),
                target_corr_dict.get(col_A, 0),
                -qvalue_dict.get(col_A, 1.0)
            )
            score_B = (
                sig_count_dict.get(col_B, 0),
                target_corr_dict.get(col_B, 0),
                -qvalue_dict.get(col_B, 1.0)
            )

            if score_A >= score_B:
                to_drop.add(col_B)
            else:
                to_drop.add(col_A)

final_common_descriptors = [col for col in X_fdr.columns if col not in to_drop]

print(f"Финальное количество единых дескрипторов: {len(final_common_descriptors)}")

# =========================================================
# 5. ИТОГОВЫЙ НАБОР ПРИЗНАКОВ
# =========================================================
X_selected_common = X[final_common_descriptors].copy()

# =========================================================
# 6. ДОП. ТАБЛИЦЫ ДЛЯ АНАЛИЗА
# =========================================================
final_feature_info = agg_results[
    agg_results["Descriptor"].isin(final_common_descriptors)
].sort_values(
    by=["Significant_Targets_Count", "MaxAbsCorrelation", "Min_FDR_q_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("\nТоп-10 итоговых признаков:")
print(final_feature_info.head(10))

Исходное количество числовых признаков: 273
Осталось после multi-target FDR union: 216
Финальное количество единых дескрипторов: 110

Топ-10 итоговых признаков:
                 Descriptor  MaxAbsCorrelation  MeanAbsCorrelation  \
0          Chi_to_RingCount           0.310238            0.231607   
1                  VSA_mean           0.289324            0.160132   
2  NumSaturatedHeterocycles           0.281584            0.211995   
3       NumValenceElectrons           0.269125            0.153113   
4          Rotatable_per_HA           0.267830            0.190382   
5                SlogP_VSA5           0.244430            0.182003   
6          VSA_EState4_x_MW           0.238416            0.177826   
7                    fr_NH2           0.237659            0.184337   
8               EState_VSA8           0.214811            0.150438   
9               EState_VSA5           0.208346            0.160079   

   Min_FDR_q_value  Significant_Targets_Count  Keep_By_Union  
0    

In [27]:
train_selected = pd.concat([X_selected_common, train[target_cols]], axis=1)
train_selected.head()

,AvgIpc,BCUT2D_CHGLO,BCUT2D_LOGPHI,BCUT2D_MRLOW,BCUT2D_MWHI,BCUT2D_MWLOW,BalabanJ,Charge_Asymmetry,Chi_Complexity,Chi_to_RingCount,...,fr_phenol,fr_piperzine,fr_quatN,fr_sulfide,fr_unbrch_alkane,fr_urea,qed,pIC50,pCC50,log10_SI
0,2.471240,-2.203512,2.178787,0.155539,32.166506,10.291948,2.063714,0.241070,2.590056,2.162989,...,0,0,0,1,0,0,0.681165,0.989639,1.018827,-0.029188
1,2.314947,-2.449813,2.456783,-0.003171,35.495692,9.631497,2.157249,0.155411,3.202665,5.979250,...,0,0,0,0,0,0,0.769122,4.353274,2.075665,2.277609
2,2.579710,-2.562182,2.584438,-0.007335,16.507764,9.486685,1.480238,0.332383,3.271705,3.649563,...,0,0,0,0,0,0,0.612606,2.352816,1.300287,1.052529
3,3.101827,-2.040425,2.409619,-0.384432,32.227747,10.102030,2.026965,0.162038,4.970214,5.111900,...,0,0,0,0,0,0,0.345823,2.165714,0.165714,2.000000
4,1.787472,-2.547053,2.539193,0.000365,14.788664,9.531931,1.940004,0.308846,3.002350,1.657054,...,0,0,0,0,0,0,0.562066,2.698264,1.154893,1.543371


In [28]:
test_selected = test[X_selected_common.columns]
test_selected.head()

,AvgIpc,BCUT2D_CHGLO,BCUT2D_LOGPHI,BCUT2D_MRLOW,BCUT2D_MWHI,BCUT2D_MWLOW,BalabanJ,Charge_Asymmetry,Chi_Complexity,Chi_to_RingCount,...,fr_ketone_Topliss,fr_methoxy,fr_nitro,fr_phenol,fr_piperzine,fr_quatN,fr_sulfide,fr_unbrch_alkane,fr_urea,qed
0,3.494848,-2.425359,2.544663,-0.147725,32.166547,9.704289,1.531269,0.173188,3.264259,3.565174,...,0,0,0,0,0,0,1,0,0,0.770057
1,2.752143,-2.390353,2.369524,-0.277225,16.707906,9.991346,1.885708,0.269334,4.373269,2.230052,...,0,0,0,3,0,0,0,0,0,0.278628
2,2.001545,-2.580818,2.603245,-0.063729,14.717158,9.477838,1.911265,0.320711,3.081114,2.190560,...,0,0,0,0,0,0,0,0,0,0.657472
3,3.093791,-2.378300,2.497185,-0.343838,19.431747,9.748466,2.061305,0.161809,4.035225,4.196103,...,1,0,0,0,0,0,0,0,0,0.564307
4,2.896060,-2.201773,2.471778,-0.097822,16.731298,9.952325,1.940561,0.093867,3.310435,2.390479,...,1,0,0,0,0,0,0,0,0,0.696213


In [29]:
train_selected.to_csv('/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/train_selected.csv', index=False)
test_selected.to_csv('/Users/irinaryzova/Desktop/ДПО_МИФИ/Хакатон/test_selected.csv', index=False)